In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lightcurvelynx.obstable.ztf_obstable import ZTFObsTable, _ztfcam_ccd_gain
from lightcurvelynx.astro_utils.mag_flux import mag2flux
import sqlite3

In [2]:
con = sqlite3.connect("data/ztf_metadata_latest.db")
sql_query = "SELECT * FROM exposures"
metadata_table = pd.read_sql_query(sql_query, con)
metadata_table = metadata_table.replace("", np.nan)
metadata_table = metadata_table.dropna(subset=["fwhm"])
metadata_table.columns

/var/folders/zs/zxl3t6ks12zg2l3dp9qn1rkr0000gn/T/ipykernel_35996/2534340399.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  metadata_table = metadata_table.replace("", np.nan)


Index(['expid', 'field', 'filter', 'obsdate', 'ra', 'dec', 'exptime',
       'airmass', 'infobits', 'dr', 'numsci', 'numdiff', 'fwhm', 'maglim',
       'scibckgnd', 'ellip', 'ellippa'],
      dtype='object')

In [3]:
obs_log = pd.read_parquet('ztfsniadr2/tables/observing_logs.parquet')
obs_log = obs_log.loc[obs_log.infobits < 33554432]

In [4]:
obs_log.head()

,mjd,band,fieldid,fieldra,fielddec,rcid,maglimit,zp,gain,expid,infobits,skynoise
0,58288.171875,ztfi,375,3.787257,-0.171915,16,19.650887,25.644886,6.2,53417042,0,49.960846
1,58288.171875,ztfi,375,3.787257,-0.171915,17,19.612961,25.651960,6.2,53417042,0,52.075066
2,58288.171875,ztfi,375,3.787257,-0.171915,18,19.686562,25.683561,6.2,53417042,0,50.099094
3,58288.171875,ztfi,375,3.787257,-0.171915,19,19.632156,25.650158,6.2,53417042,0,51.077618
4,58288.171875,ztfi,375,3.787257,-0.171915,20,19.573307,25.693308,6.2,53417042,0,56.108715


In [5]:
obs_log_group_by_expid = obs_log.groupby('expid')

In [6]:
metadata_group_by_expid = metadata_table.groupby('expid')

In [13]:
metadata_table.loc[metadata_table.expid == 89044068]

,expid,field,filter,obsdate,ra,dec,exptime,airmass,infobits,dr,numsci,numdiff,fwhm,maglim,scibckgnd,ellip,ellippa
211290,89044068,442,g,2019-06-10 10:34:35.291,326.5625,-2.65,30.0,1.383,0,2,64,0,2.347755,20.80725,95.8358,0.034969,-2.990865


In [14]:
obs_log.loc[obs_log.expid == 89044068]

,mjd,band,fieldid,fieldra,fielddec,rcid,maglimit,zp,gain,expid,infobits,skynoise
10344295,58644.441406,ztfg,442,5.699591,-0.046251,18,21.311857,26.125858,6.2,89044068,0,16.851185
10344296,58644.441406,ztfg,442,5.699591,-0.046251,19,21.297340,26.103340,6.2,89044068,0,16.727455
10344297,58644.441406,ztfg,442,5.699591,-0.046251,20,21.277397,26.164396,6.2,89044068,0,18.023106
10344298,58644.441406,ztfg,442,5.699591,-0.046251,21,21.292517,26.124018,6.2,89044068,0,17.124992
10344299,58644.441406,ztfg,442,5.699591,-0.046251,22,21.300348,26.183849,6.2,89044068,0,17.965132
...,...,...,...,...,...,...,...,...,...,...,...,...
10344354,58644.441406,ztfg,442,5.699591,-0.046251,43,21.294992,26.195993,6.2,89044068,0,18.257040
10344355,58644.441406,ztfg,442,5.699591,-0.046251,44,21.286552,26.208553,6.2,89044068,0,18.613600
10344356,58644.441406,ztfg,442,5.699591,-0.046251,45,21.141266,26.189264,6.2,89044068,0,20.903999
10344357,58644.441406,ztfg,442,5.699591,-0.046251,46,21.230799,26.177799,6.2,89044068,0,19.047159


In [15]:
obs_log.loc[obs_log.expid == 89044068,["maglimit","zp","skynoise"]].mean()

maglimit    21.274260
zp          26.149691
skynoise    17.849115
dtype: float32

In [16]:
obs_log.loc[obs_log.expid == 53417042,["maglimit","zp","skynoise"]].std()

maglimit    0.121340
zp          0.129181
skynoise    7.952237
dtype: float32

In [12]:
avg_obs_log = obs_log_group_by_expid[["zp","maglimit","skynoise"]].mean(numeric_only=True)
std_obs_log = obs_log_group_by_expid[["zp","maglimit","skynoise"]].std(numeric_only=True)
avg_metadata = metadata_group_by_expid[["maglim","scibckgnd","fwhm"]].mean(numeric_only=True)
std_metadata = metadata_group_by_expid[["maglim","scibckgnd","fwhm"]].std(numeric_only=True)

In [30]:
avg_obs_log

,zp,maglimit,skynoise
expid,,,
53417042,25.592709,19.451427,57.750450
53417087,25.597589,19.491785,55.883972
53417134,25.594849,19.735231,44.463055
53417179,25.603411,20.128443,31.202841
53417225,25.618179,20.466686,23.333241
...,...,...,...
151955548,26.115932,18.273958,274.991547
151955596,26.092585,18.268255,270.373047
151955651,26.099012,18.155870,303.747131


In [31]:
avg_metadata

,maglim,scibckgnd,fwhm
expid,,,
44312385,20.59370,1584.2450,3.186245
44313564,20.28910,1339.1800,4.072620
44314192,20.10565,1680.4250,4.236985
44316126,18.88630,119.3450,3.699770
44316172,18.65080,110.0920,3.986960
...,...,...,...
249354630,17.98250,454.6340,4.566850
249354677,18.03450,313.1560,4.758450
249354724,18.36850,255.3100,3.788825


In [32]:
std_obs_log

,zp,maglimit,skynoise
expid,,,
53417042,0.129181,0.121340,7.952237
53417087,0.129541,0.130156,7.540013
53417134,0.129592,0.114734,5.373348
53417179,0.131351,0.138408,3.830918
53417225,0.162603,0.149761,4.775217
...,...,...,...
151955548,0.068976,0.080273,24.103956
151955596,0.076824,0.100852,20.455225
151955651,0.094475,0.187652,41.665577
